In [2]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from sklearn.metrics import roc_auc_score

In [36]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import ttest_ind, chi2_contingency

In [4]:
def fast_vif_numpy(df, feature_cols):
    # Impute missing values first (or drop them)
    X = df[feature_cols].dropna() 
    
    # Calculate the correlation matrix
    corr_matrix = X.corr().values
    
    # Invert the matrix (this calculates all VIFs simultaneously)
    try:
        inverse_matrix = np.linalg.inv(corr_matrix)
        vifs = np.diag(inverse_matrix)
    except np.linalg.LinAlgError:
        # If features are PERFECTLY collinear, inversion fails
        return "Error: Perfect multicollinearity detected. Drop identical columns first."
        
    vif_df = pd.DataFrame({'feature': feature_cols, 'VIF': vifs})
    return vif_df.sort_values(by='VIF', ascending=False).reset_index(drop=True)

In [5]:
def fast_vif_clean(df, feature_cols):
    # 1. You MUST handle missing values completely first
    all_nan_cols = df[feature_cols].columns[df[feature_cols].isna().all()]

    constant_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]

    feature_cols = [c for c in feature_cols if c not in all_nan_cols.to_list() + constant_cols]

    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(df[feature_cols]), columns=feature_cols)
    
    # 2. Get the clean correlation matrix
    corr_matrix = X_imputed.corr()
    
    # 3. Use PINV (Pseudo-Inverse) to safely handle near-singular matrices
    inverse_matrix = np.linalg.pinv(corr_matrix.values)
    vifs = np.diag(inverse_matrix)
    
    # 4. Map back to a sorted DataFrame
    vif_df = pd.DataFrame({'feature': feature_cols, 'VIF': vifs})
    return vif_df.sort_values(by='VIF', ascending=False).reset_index(drop=True)

In [6]:
def target_correlation_value(df, feature_cols, target_col, corr_threshold=0.60):
    """
    Step-by-step Correlation with target.
    
    Args:
        df (pd.DataFrame): The input dataframe
        feature_cols (list): List of feature names to analyze
        target_col (str): The name of your target column (used to break ties in correlation)
        corr_threshold (float): Pairs with correlation above this will have the weaker feature dropped
        
    Returns:
        list: A trimmed down, stable list of feature names ready for LightGBM
    """
    print(f"Initial Feature Count: {len(feature_cols)}")

    target_corr = df[feature_cols].apply(lambda col: df[target_col].corr(col)).abs()
    target_corr = target_corr.reset_index()
    target_corr.columns = ['feature', 'target_corr']
    return(target_corr)

In [7]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
#from statsmodels.stats.outliers_influence import variance_inflation_factor

def analyze_and_filter_multicollinearity(df, feature_cols, target_col, corr_threshold=0.85, vif_threshold=10.0):
    """
    Step-by-step multicollinearity filter using Correlation and VIF.
    
    Args:
        df (pd.DataFrame): The input dataframe
        feature_cols (list): List of feature names to analyze
        target_col (str): The name of your target column (used to break ties in correlation)
        corr_threshold (float): Pairs with correlation above this will have the weaker feature dropped
        vif_threshold (float): Features with a VIF above this will be iteratively dropped
        
    Returns:
        list: A trimmed down, stable list of feature names ready for LightGBM
    """
    print(f"Initial Feature Count: {len(feature_cols)}")
    
    # ----------------------------------------------------
    # PHASE 1: PAIRWISE CORRELATION FILTER
    # ----------------------------------------------------
    corr_matrix = df[feature_cols].corr().abs()
    target_corr = df[feature_cols].apply(lambda col: df[target_col].corr(col)).abs()
    
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop_corr = set()
    
    for col in upper_tri.columns:
        highly_corr = upper_tri.index[upper_tri[col] > corr_threshold].tolist()
        if highly_corr:
            cluster = highly_corr + [col]
            best_feature = target_corr[cluster].idxmax()
            for f in cluster:
                if f != best_feature:
                    to_drop_corr.add(f)
                    
    current_features = [c for c in feature_cols if c not in to_drop_corr]
    print(f"-> Phase 1 (Correlation): Dropped {len(to_drop_corr)} features. {len(current_features)} remaining.")
    
    # ----------------------------------------------------
    # PHASE 2: ITERATIVE VIF FILTER
    # ----------------------------------------------------
    # VIF cannot handle missing (NaN) values, so we must impute them first
    X = df[current_features].copy()
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=current_features)
    
    # We must add an explicit constant/intercept column because VIF calculations 
    # require an intercept to calculate the R-squared values accurately under the hood
    X_imputed['_constant'] = 1.0
    
    dropped_via_vif = []
    
    while True:
        # 1. Re-calculate VIF scores for the remaining features in the loop
        vif_data = pd.DataFrame()
        vif_data["feature"] = [col for col in X_imputed.columns if col != '_constant']
        
        # Calculate VIF score for each feature index position
        vif_scores = []
        for i in range(X_imputed.shape[1]):
            col_name = X_imputed.columns[i]
            if col_name != '_constant':
                # Map back column position index to calculate VIF
                score = variance_inflation_factor(X_imputed.values, i)
                vif_scores.append(score)
                
        vif_data["VIF"] = vif_scores
        
        # 2. Find the worst offender
        max_vif = vif_data["VIF"].max()
        
        if max_vif > vif_threshold:
            # Identify which feature has that maximum VIF score
            worst_feature = vif_data.loc[vif_data["VIF"].idxmax(), "feature"]
            
            # Drop it from our running tracking matrix
            X_imputed = X_imputed.drop(columns=[worst_feature])
            dropped_via_vif.append(worst_feature)
        else:
            # If no features violate our threshold max value, we break out of the loop
            break
            
    final_features = [c for c in current_features if c not in dropped_via_vif]
    print(f"-> Phase 2 (VIF): Dropped {len(dropped_via_vif)} multi-collinear features iteratively.")
    print(f"Final Count for LightGBM: {len(final_features)}\n")
    
    return final_features

In [8]:
def ttest_one_var(test_data, feature_col, target_col, equal_var=False):
    """
    Perform an independent t-test for one numeric column against a binary target.
    
    Args:
        test_data (pd.dataframe): Numeric data to test
        feature_col (column name of feature): Numeric feature to test
        target_col (column name of target): Binary target (must have exactly 2 groups)  
        equal_var (bool): If False (default), uses Welch's t-test (unequal variances)
    
    Returns:
        dict: t-statistic, p-value, group means, and effect size (Cohen's d)
    
    Example:
        results = ttest_one_var(df, 'LTV', 'TARGET', equal_var=False)
    """
    # Check for exactly 2 groups
    groups = [0, 1]
    
    # Split data into two groups
    group1 = test_data.loc[test_data[target_col] == groups[0], feature_col].dropna()
    group2 = test_data.loc[test_data[target_col] == groups[1], feature_col].dropna()
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=equal_var)
    
    # Calculate effect size (Cohen's d)
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1+n2-2))
    cohens_d = (group1.mean() - group2.mean()) / pooled_std
    
    return {
        'feature': feature_col,
        't_statistic': t_stat,
        'p_value': p_value,
        'mean_group1': group1.mean(),
        'mean_group2': group2.mean(),
        'cohens_d': cohens_d,
        'abs_t_stat': abs(t_stat)  # Useful for ranking variables
    }

In [9]:
def calculate_auc_one_var(df, numeric_col, target_col):
    """
    Simplified AUC calculation using sklearn's roc_auc_score.
    Returns json with AUC scores.
    """
    x = df[numeric_col]
    y = df[target_col]
    valid_idx = x.notna() & y.notna()  # Drop missing values

    try:
        auc = roc_auc_score(y[valid_idx], x[valid_idx])
        gini_abs = abs(2 * auc - 1)  # Convert AUC to Gini coefficient
        result = {'variable': numeric_col, 'auc_score': auc, 'gini_abs': gini_abs}
    except ValueError:
        result = {'variable': numeric_col, 'auc_score': None, 'gini_abs': None}
    
    return result

In [40]:
def calculate_numeric_selection_metrics(df, numeric_cols, cat_cols = None, target_col='TARGET'):
    """
    Calculate AUC and t-test for a numeric feature against a binary target.
    
    Args:
        df (pd.DataFrame): DataFrame containing the feature and target
        numeric_cols (list): List of numeric feature column names
        cat_cols (list): List of categorical feature column names
        target_col (str): Name of the binary target column
    
    Returns:
        dict: AUC score and t-test results
    """

    results = []
    for col in numeric_cols:
        auc_result = calculate_auc_one_var(df, col, target_col)
        ttest_result = ttest_one_var(df, col, target_col, equal_var=False)
        missing_perc = df[col].isna().sum()/df.shape[0] * 100
        results.append({
            'variable': col,
            'missing_count': missing_perc,
            'auc_score': auc_result['auc_score'],
            'gini_abs': auc_result['gini_abs'],
            't_statistic': ttest_result['t_statistic'],
            'p_value': ttest_result['p_value'],
            'mean_group1': ttest_result['mean_group1'],
            'mean_group2': ttest_result['mean_group2'],
            'cohens_d': ttest_result['cohens_d']
        })
    if cat_cols is not None:
        for col in cat_cols:
            contingency_table = pd.crosstab(df[col], df[target_col])
            # Run Chi-Square test
            stat, p_value, dof, expected = chi2_contingency(contingency_table)
            missing_perc = df[col].isna().sum()/df.shape[0] * 100
            results.append({
                'variable': col,
                'missing_count': missing_perc,
                'auc_score': None,
                'gini_abs': None,
                't_statistic': None,
                'p_value': p_value,
                'mean_group1': None,
                'mean_group2': None,
                'cohens_d': None
            })
    return pd.DataFrame(results)
    

In [11]:
def multivariate_logistic_regression_coef(df, feature_cols, target_col):

    X = df[feature_cols]
    y = df[target_col]

    # Loop through each feature one at a time
    for col in feature_cols:
        # Extract a single feature as a 2D array/DataFrame for scikit-learn
        X_single = df[[col]]
        
        # 1. Handle missing values
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X_single)

        # Check if the column is binary (contains only 0, 1, or a single unique value of 0 or 1)
        unique_vals = np.unique(X_imputed)
        is_binary = set(unique_vals).issubset({0.0, 1.0, 0, 1})

        # 2. Set up conditional scaling
        if is_binary:
            # Pass through untouched if it's binary
            X_final = X_imputed
        else:
            # Scale if it's continuous
            scaler = StandardScaler()
            X_final = scaler.fit_transform(X_imputed)

        df[[col]] = X_final  # Update the original DataFrame with the processed feature

    # Fit logistic regression model
    model = LogisticRegression(random_state=123, max_iter=10000, penalty='l1', solver='liblinear', C=0.1)
    model.fit(df[feature_cols], y)

    # Create DataFrame with coefficients
    coef_df = pd.DataFrame({'variable': feature_cols, 'coef': model.coef_[0]})

    return coef_df

In [12]:

def univariate_logistic_regression_coef(df, feature_cols, target_col):
    """
    Fit a separate logistic regression model for each individual feature 
    and return their coefficients.
    
    Args:
        df (pd.DataFrame): DataFrame containing features and target
        feature_cols (list): List of feature column names
        target_col (str): Name of the target column
    
    Returns:
        pd.DataFrame: DataFrame with feature names and their single-variable coefficients
    """
    y = df[target_col]
    results = []

    # Loop through each feature one at a time
    for col in feature_cols:
        # Extract a single feature as a 2D array/DataFrame for scikit-learn
        X_single = df[[col]]
        
        # 1. Handle missing values
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X_single)

        # Check if the column is binary (contains only 0, 1, or a single unique value of 0 or 1)
        unique_vals = np.unique(X_imputed)
        is_binary = set(unique_vals).issubset({0.0, 1.0, 0, 1})

        # 2. Set up conditional scaling
        if is_binary:
            # Pass through untouched if it's binary
            X_final = X_imputed
        else:
            # Scale if it's continuous
            scaler = StandardScaler()
            X_final = scaler.fit_transform(X_imputed)

        # Fit logistic regression model on this single feature
        model = LogisticRegression(random_state=123, max_iter=10000, penalty='l1', solver='liblinear', C=0.1)
        model.fit(X_final, y)
        
        # Extract the coefficient
        coef_value = model.coef_[0][0]
        results.append({'variable': col, 'coef': coef_value})

    # Combine all individual results into a single DataFrame
    coef_df = pd.DataFrame(results)
    
    return coef_df

In [43]:
def calculate_all_numeric_selection_metrics(df, numeric_cols, cat_cols = None, target_col = 'TARGET'):
    """
    Calculate all selection metrics for numeric features and return a combined DataFrame.
    
    Args:
        df (pd.DataFrame): DataFrame containing features and target
        numeric_cols (list): List of numeric feature column names
        cat_cols (list): List of categorical feature column names
        target_col (str): Name of the target column
    Returns:
        pd.DataFrame: Combined DataFrame with all selection metrics for numeric features
    """
    # Calculate AUC and t-test metrics
    auc_ttest = calculate_numeric_selection_metrics(df, numeric_cols, cat_cols=cat_cols, target_col=target_col)
    
    # Calculate univariate logistic regression coefficients
    coef_df = multivariate_logistic_regression_coef(df, numeric_cols, target_col)
    
    # Calculate VIF scores
    vif_df = fast_vif_clean(df, numeric_cols)
    
    # Calculate correlation with target
    target_corr_df = target_correlation_value(df, numeric_cols, target_col)
    
    # Merge all metrics into a single DataFrame
    cat_cols = cat_cols if cat_cols is not None else []
    numeric_selection_metrics = pd.DataFrame({'variable': numeric_cols + cat_cols})
    numeric_selection_metrics = numeric_selection_metrics.merge(auc_ttest, how='left', right_on='variable', left_on='variable')
    numeric_selection_metrics = numeric_selection_metrics.merge(coef_df, how='left', right_on='variable', left_on='variable')
    numeric_selection_metrics = numeric_selection_metrics.merge(vif_df, how='left', right_on='feature', left_on='variable').drop(columns=['feature'])
    numeric_selection_metrics = numeric_selection_metrics.merge(target_corr_df, how='left', right_on='feature', left_on='variable').drop(columns=['feature'])
    
    return numeric_selection_metrics

In [14]:
numeric_selection_metrics_list = list()

In [15]:
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit')

# Credit Card

In [198]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [199]:
df_with_target = pd.read_parquet('src\\intermediate\\creditcard_processed.parquet')

In [200]:
numcols = df_with_target.select_dtypes(include=np.number).columns.to_list()
numcols.remove('TARGET')

In [201]:
len(numcols)

372

In [202]:
[col for col in df_with_target if col not in numcols]

['SK_ID_CURR', 'TARGET']

In [203]:
metrics_cc = calculate_all_numeric_selection_metrics(df_with_target, numcols, 'TARGET')
metrics_cc['Dataset'] = 'CreditCard'

Initial Feature Count: 372


In [204]:
numeric_selection_metrics_list.append(metrics_cc)

# Payment Installments Data

In [205]:
df_with_target = pd.read_parquet('src\\intermediate\\payment_installments_processed.parquet')

In [206]:
#df_with_target = df_with_target.merge(base, how='left', on='SK_ID_CURR')

In [207]:
df_with_target.columns

Index(['SK_ID_CURR', 'PMNT_INSTLMT_VRSN_CHANGES_COUNT_MEAN',
       'PMNT_INSTLMT_VRSN_CHANGES_COUNT_MAX',
       'PMNT_INSTLMT_VRSN_CHANGES_COUNT_MIN',
       'PMNT_INSTLMT_DURATION_DAYS_MEAN', 'PMNT_INSTLMT_DURATION_DAYS_MAX',
       'PMNT_INSTLMT_DURATION_DAYS_MIN',
       'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_MEAN',
       'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_MAX',
       'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_MIN',
       'PMNT_INSTLMT_NUM_INSTALMENTS_MEAN', 'PMNT_INSTLMT_NUM_INSTALMENTS_MAX',
       'PMNT_INSTLMT_NUM_INSTALMENTS_MIN',
       'PMNT_INSTLMT_VRSN_CHANGES_COUNT_L1Y_MEAN',
       'PMNT_INSTLMT_VRSN_CHANGES_COUNT_L1Y_MAX',
       'PMNT_INSTLMT_VRSN_CHANGES_COUNT_L1Y_MIN',
       'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_L1Y_MEAN',
       'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_L1Y_MAX',
       'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_L1Y_MIN',
       'PMNT_INSTLMT_AMT_PAYMENT_MEAN_1Y', 'PMNT_INSTLMT_AMT_PAYMENT_MAX_1Y',
       'PMNT_INSTLMT_AMT_PAYMENT_MIN_1Y',
       'PMNT_INSTLMT_AMT_INSTALMEN

In [208]:
numcols = df_with_target.select_dtypes(include=np.number).columns.to_list()
numcols.remove('TARGET')

In [209]:
[col for col in df_with_target if col not in numcols]

['SK_ID_CURR', 'TARGET']

In [210]:
metrics_installments = calculate_all_numeric_selection_metrics(df_with_target, numcols, 'TARGET')
metrics_installments['Dataset'] = 'Payment Installments'

Initial Feature Count: 70


In [211]:
numeric_selection_metrics_list.append(metrics_installments)

# POS Data

In [212]:
df_with_target = pd.read_parquet('src\\intermediate\\posdata_processed.parquet')

In [213]:
numcols = df_with_target.select_dtypes(include=np.number).columns.to_list()
numcols.remove('TARGET')

In [214]:
[col for col in df_with_target if col not in numcols]

['SK_ID_CURR', 'TARGET']

In [215]:
numcols

['POS_DAYS_IN_DPD',
 'POS_DAYS_IN_DPD_DEF',
 'POS_DAYS_IN_DPD_TO_DURATION_RATIO',
 'POS_DAYS_IN_DPD_1Y',
 'POS_DAYS_IN_DPD_DEF_1Y',
 'POS_DAYS_IN_DPD_TO_DURATION_RATIO_1Y',
 'POS_IS_IN_LAST_1Y',
 'POS_DAYS_IN_DPD_3M',
 'POS_DAYS_IN_DPD_DEF_3M',
 'POS_DAYS_IN_DPD_TO_DURATION_RATIO_3M',
 'POS_IS_IN_LAST_3M',
 'POS_AVG_CNT_INSTALMENT_LAST_MONTH',
 'POS_AVG_CNT_INSTALMENT_FUTURE_LAST_MONTH',
 'POS_MAX_CNT_INSTALMENT_LAST_MONTH',
 'POS_MAX_CNT_INSTALMENT_FUTURE_LAST_MONTH',
 'POS_CNT_ACTIVE_PREV_CONTRACT_STATUS_LAST_MONTH',
 'POS_SK_DPD_LAST_MONTH',
 'POS_SK_DPD_DEF_LAST_MONTH',
 'POS_DAYS_SINCE_LAST_DPD_GT_1',
 'POS_TOTAL_PREV_LOAN',
 'POS_SUM_PREV_CREDIT_PREPAYMENT_FLAG',
 'POS_AVG_CREDIT_TO_APPL',
 'POS_ANNUITY_TO_CREDIT_RATIO',
 'POS_CNT_HIGH_YIELD_GROUP',
 'POS_CNT_MIDDLE_YIELD_GROUP',
 'POS_CNT_NFLAG_INSURED_ON_APPROVAL_PREVLOAN',
 'POS_AVG_DAYS_DECISION_PREVLOAN']

In [216]:
metrics_pos = calculate_all_numeric_selection_metrics(df_with_target, numcols, 'TARGET')
metrics_pos['Dataset'] = 'POS Data'

Initial Feature Count: 27


In [217]:
numeric_selection_metrics_list.append(metrics_pos)

# Burea Data

In [ ]:
df_with_target = pd.read_parquet('src\\intermediate\\bureau_processed.parquet')

In [219]:
df_with_target = df_with_target.merge(base, on='SK_ID_CURR', how='inner')

In [220]:
numcols = df_with_target.select_dtypes(include=np.number).columns.to_list()
numcols.remove('TARGET')

In [221]:
numcols

['BUREAU_TOTAL_CREDIT_COUNT',
 'BUREAU_CREDIT_TYPE_TOTAL_COUNT',
 'BUREAU_CONSUMERCREDIT_COUNT',
 'BUREAU_CREDITCARD_COUNT',
 'BUREAU_CARLOAN_COUNT',
 'BUREAU_MORTGAGE_COUNT',
 'BUREAU_TOTAL_ACTIVE_CREDIT_COUNT',
 'BUREAU_PAST_BAD_CREDIT_COUNT',
 'BUREAU_PAST_SOLD_CREDIT_COUNT',
 'BUREAU_DAYS_TO_LAST_CREDIT',
 'BUREAU_AVG_DAYS_TO_CREDIT',
 'BUREAU_AVG_DURATION',
 'BUREAU_MAX_DAYS_CREDIT_END',
 'BUREAU_IS_PAST_OVERDUE_FLAG',
 'BUREAU_ACTIVE_UTILIZATION_RATIO',
 'BUREAU_MAX_CREDIT_OVERDUE_AMOUNT',
 'BUREAU_AVG_CREDIT_AMOUNT',
 'BUREAU_AVG_CREDIT_DEBT_AMOUNT',
 'BUREAU_OVERDRAFT_FLAG_COUNT',
 'BUREAU_AVG_ANNUITY_PERCENT',
 'BUREAUBAL_LAST_STATUS_DPD_GT_90_ID_FLAG',
 'BUREAUBAL_LAST_STATUS_DPD_GT_90_ID_COUNT',
 'BUREAUBAL_DAYS_IN_DPD_GT_90_ID_COUNT',
 'BUREAUBAL_DAYS_IN_DPD_GT_90_DAYS_COUNT',
 'BUREAUBAL_DAYS_IN_DPD_GT_0_ID_COUNT',
 'BUREAUBAL_DAYS_IN_DPD_GT_0_DAYS_COUNT']

In [222]:
metrics_buro = calculate_all_numeric_selection_metrics(df_with_target, numcols, 'TARGET')
metrics_buro['Dataset'] = 'Bureau Data'

C:\Users\domjo\AppData\Local\Temp\ipykernel_19160\46911737.py:30: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std


Initial Feature Count: 26


a:\Dominic\My Projects\Home Credit\project_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
a:\Dominic\My Projects\Home Credit\project_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [223]:
numeric_selection_metrics_list.append(metrics_buro)

# Application data

In [28]:
df_with_target = pd.read_parquet("src\\intermediate\\application_train_data_processed.parquet")

In [29]:
numcols = df_with_target.select_dtypes(include=np.number).columns.to_list()
numcols.remove('TARGET')

In [30]:
numcols

['FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',
 'LANDAREA_AVG',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVINGAREA_AVG',
 'APARTMENTS_MODE',
 

In [46]:
cat_cols = df_with_target.select_dtypes(include=['object']).columns.to_list()

In [53]:
cat_cols

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [47]:
metrics_appl = calculate_all_numeric_selection_metrics(df_with_target, numcols, cat_cols=cat_cols, target_col='TARGET')
metrics_appl['Dataset'] = 'Application Data'

a:\Dominic\My Projects\Home Credit\project_env\Lib\site-packages\scipy\stats\_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Initial Feature Count: 125


In [49]:
numeric_selection_metrics_list.append(metrics_appl)

Combine all results

In [50]:
numeric_selection_metrics_full_df = pd.concat(numeric_selection_metrics_list, axis=0, ignore_index=True)

In [51]:
numeric_selection_metrics_full_df['points'] = (
    (numeric_selection_metrics_full_df['coef'] > 0 )* 1 +\
    ( numeric_selection_metrics_full_df['gini_abs'] > 0.1 )* 1 +\
    ( numeric_selection_metrics_full_df['p_value'] < 0.05 )* 1 +\
    ( abs(numeric_selection_metrics_full_df['cohens_d']) > 0.4 )* 1 +\
    ( numeric_selection_metrics_full_df['VIF'] < 10 )* 1 +\
    ( numeric_selection_metrics_full_df['target_corr'] > 0.5 )* 1
)

In [52]:
numeric_selection_metrics_full_df.to_excel('src\\Variable-Selection\\numeric_selection_metrics_full_df_3.xlsx', index=False)